# 02 — Feature Analysis: Human vs. AI

This notebook visualises the stylometric feature distributions for human-written
vs. AI-generated code and performs SHAP analysis on the statistical baseline model.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from config.settings import MODELS_DIR, SPLITS_DIR
from src.models.statistical_baseline import build_feature_matrix, get_feature_columns

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline

In [ ]:
# Load test set and extract features
test_df = pd.read_parquet(SPLITS_DIR / 'test.parquet')
test_feat = build_feature_matrix(test_df)
feature_cols = get_feature_columns(test_feat)
print(f'Feature columns: {feature_cols}')

In [ ]:
# Feature distributions: human vs AI
plot_features = feature_cols[:12]  # plot first 12 features
n_cols = 3
n_rows = (len(plot_features) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(plot_features):
    for label, name in [(0, 'Human'), (1, 'AI')]:
        vals = test_feat[test_feat['label'] == label][feat].dropna()
        axes[i].hist(vals, bins=30, alpha=0.6, label=name)
    axes[i].set_title(feat)
    axes[i].legend()

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Per-language feature comparison
for lang in ['cpp', 'python', 'java']:
    subset = test_feat[test_df['language'] == lang] if 'language' in test_df.columns else test_feat
    if len(subset) == 0:
        continue
    print(f'\n=== {lang.upper()} ===')
    for feat in feature_cols[:6]:
        human_mean = subset[subset['label'] == 0][feat].mean()
        ai_mean = subset[subset['label'] == 1][feat].mean()
        print(f'  {feat}: human={human_mean:.3f}, ai={ai_mean:.3f}')

In [ ]:
# SHAP analysis on the XGBoost baseline
model_path = MODELS_DIR / 'xgb_baseline.pkl'
if model_path.exists():
    with open(model_path, 'rb') as f:
        model = pickle.load(f)

    X_test = test_feat[feature_cols].fillna(0)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values, X_test, feature_names=feature_cols, show=False)
    plt.tight_layout()
    plt.show()
else:
    print('No trained model found. Run statistical_baseline.py first.')

In [ ]:
# Correlation matrix of features
corr = test_feat[feature_cols].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=feature_cols, yticklabels=feature_cols)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()